# Cross-Encoder Training (Model 1)

Trains corpus-specific cross-encoders using ListNet loss on E5 + TF-IDF retrieved candidates.

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

## Step 2: Build Training Groups

Retrieves candidates for each query using 3-way RRF fusion:
- **E5** (multilingual-e5-large): Dense semantic retrieval
- **TF-IDF (word)**: Unigram + bigram sparse retrieval  
- **TF-IDF (char)**: Character 3-4gram sparse retrieval

Default weights: E5=0.6, TF-IDF(word)=0.4, TF-IDF(char)=0.15

Outputs train/val splits as JSONL with query, candidate passages, and relevance labels.

In [3]:
# === Stage 1: Build fused (E5 ⊕ TF-IDF via RRF) groups and save them ===
import os, json, math, time, random, hashlib, gc
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------- Config ----------------------
CORPUS_JSONL   = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")
TRAIN_JSONL    = os.getenv("TRAIN_JSONL",  "/content/hsrc_train_augmented.jsonl")
OUTPUT_DIR     = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")

E5_MODEL_NAME  = os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large")  # local path or HF id

# --- Retrieval sizes ---
K_E5           = int(os.getenv("K_E5", "70"))     # final K used to build training groups (after fusion)
# RRF will search a larger pool for each source then fuse to top-K_E5:
RRF_POOL_MULT  = float(os.getenv("RRF_POOL_MULT", "3"))   # pool = max(K_E5 * mult, min)
RRF_POOL_MIN   = int(os.getenv("RRF_POOL_MIN", "150"))
RRF_K          = int(os.getenv("RRF_K", "60"))            # RRF stabilizer

# --- Fusion weights ---
WEIGHT_E5            = float(os.getenv("WEIGHT_E5", "0.6"))
WEIGHT_TFIDF         = float(os.getenv("WEIGHT_TFIDF", "0.4"))   # word-level TF-IDF
WEIGHT_TFIDF_CHAR    = float(os.getenv("WEIGHT_TFIDF_CHAR", "0.15"))  # NEW: char-level TF-IDF weight

# --- TF-IDF options (word-level only, no BM25) ---
TFIDF_MAX_FEATS = int(os.getenv("TFIDF_MAX_FEATS", "300000"))
TFIDF_NGRAM_MIN = int(os.getenv("TFIDF_NGRAM_MIN", "1"))
TFIDF_NGRAM_MAX = int(os.getenv("TFIDF_NGRAM_MAX", "2"))

# --- Char TF-IDF (identical to previous code style) ---
ENABLE_CHAR_TFIDF     = int(os.getenv("ENABLE_CHAR_TFIDF", "1"))  # 1=on, 0=off
TFIDF_CHAR_MIN        = int(os.getenv("TFIDF_CHAR_MIN", "3"))
TFIDF_CHAR_MAX        = int(os.getenv("TFIDF_CHAR_MAX", "4"))
TFIDF_CHAR_MAX_FEATS  = int(os.getenv("TFIDF_CHAR_MAX_FEATS", "200000"))
TFIDF_CHAR_ANALYZER   = os.getenv("TFIDF_CHAR_ANALYZER", "char_wb")  # 'char' or 'char_wb'

VAL_SIZE       = float(os.getenv("VAL_SIZE", "0.15"))
SEED           = int(os.getenv("SEED", "42"))

# Behavior toggles
SPLIT_VAL             = int(os.getenv("SPLIT_VAL", "1"))              # 1=create val split, 0=train on all
FILTER_VAL_LEAKAGE    = int(os.getenv("FILTER_VAL_LEAKAGE", "0"))      # 1=remove val docs from train, 0=keep (submission-style)

# Embedding cache
EMB_CACHE_DIR        = os.getenv("EMB_CACHE_DIR", "/content/e5_cache")
USE_MEMMAP_ON_LOAD   = bool(int(os.getenv("USE_MEMMAP_ON_LOAD", "1")))
Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Where to save stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(STAGE1_DIR).mkdir(parents=True, exist_ok=True)
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k{K_E5}.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k{K_E5}.jsonl"))
META_JSON          = os.getenv("STAGE1_META_JSON",   os.path.join(STAGE1_DIR, "meta.json"))

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] device={device}  K_FINAL={K_E5}  VAL_SIZE={VAL_SIZE}  SEED={SEED}  "
      f"SPLIT_VAL={SPLIT_VAL}  FILTER_VAL_LEAKAGE={FILTER_VAL_LEAKAGE}")
print(f"[PATHS] CORPUS={CORPUS_JSONL}  TRAIN={TRAIN_JSONL}  OUT={OUTPUT_DIR}  CACHE={EMB_CACHE_DIR}  STAGE1={STAGE1_DIR}")
print(f"[FUSION] RRF_POOL_MULT={RRF_POOL_MULT}  RRF_POOL_MIN={RRF_POOL_MIN}  RRF_K={RRF_K}  "
      f"WEIGHTS: E5={WEIGHT_E5} TFIDF(word)={WEIGHT_TFIDF} TFIDF(char)={WEIGHT_TFIDF_CHAR}")
print(f"[TFIDF(word)] max_features={TFIDF_MAX_FEATS} ngram=({TFIDF_NGRAM_MIN},{TFIDF_NGRAM_MAX})")
print(f"[TFIDF(char)] enabled={bool(ENABLE_CHAR_TFIDF)} analyzer={TFIDF_CHAR_ANALYZER} "
      f"ngram=({TFIDF_CHAR_MIN},{TFIDF_CHAR_MAX}) max_features={TFIDF_CHAR_MAX_FEATS}")

# ---------------------- IO ----------------------
def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid: corpus[uid] = o.get("passage") or o.get("text") or ""
    return corpus

def _as_int(x):
    if x is None: return 0
    s=str(x);  return int(s) if s.isdigit() else int(float(s))

def load_train(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            q   = o.get("query","")
            qid = o.get("query_uuid") or None
            case = o.get("case_name")  # keep this
            paras = o.get("paragraphs",{}) or {}
            labels= o.get("target_actions",{}) or {}
            gt = {}
            for i in range(1000):
                pk, lk = f"paragraph_{i}", f"target_action_{i}"
                if pk not in paras or lk not in labels: break
                puid = paras[pk].get("uuid") or paras[pk].get("id")
                rel  = _as_int(labels[lk])
                if puid: gt[puid] = rel
            rows.append({"query_uuid": qid, "query": q, "gt": gt, "case_name": case})
    return rows

corpus = load_corpus(CORPUS_JSONL)
print(f"[DATA] corpus docs={len(corpus):,}")
train_rows = load_train(TRAIN_JSONL)
print(f"[DATA] queries total={len(train_rows):,}")

# ---------------------- E5 retriever ----------------------
class E5Retriever:
    def __init__(self, name):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None),
                attn_implementation="sdpa"
            ).to(device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None)
            ).to(device)
        self.model.eval()

    @torch.no_grad()
    def embed(self, texts: List[str], is_query=False, bs=128) -> np.ndarray:
        pref = "query: " if is_query else "passage: "
        out = []
        for i in range(0, len(texts), bs):
            enc = self.tok([pref+t for t in texts[i:i+bs]], padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(device)
            h = self.model(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1)
            emb = (h*m).sum(1) / m.sum(1).clamp(min=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            out.append(emb.cpu())
        return torch.cat(out,0).numpy()

# ---------------------- Embedding cache helpers ----------------------
def _file_sig(p: Path):
    try:
        st = p.stat()
        return f"{p.name}|{st.st_size}|{int(st.st_mtime)}"
    except Exception:
        return f"{p.name}|0|0"

def _emb_cache_key(corpus_path: str, model_name: str, max_len: int, num_docs_hint: int = 0) -> str:
    p = Path(corpus_path)
    base = f"{_file_sig(p)}|{model_name}|L{max_len}|N{num_docs_hint}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:16]

def _emb_cache_paths(key: str):
    base = f"e5_{key}"
    e = os.path.join(EMB_CACHE_DIR, base + "_embeddings.npy")
    i = os.path.join(EMB_CACHE_DIR, base + "_ids.json")
    m = os.path.join(EMB_CACHE_DIR, base + "_meta.json")
    fx = os.path.join(EMB_CACHE_DIR, base + "_index.faiss")
    return e, i, m, fx

def load_embeddings_cache(corpus_path: str, model_name: str, max_len: int, expect_n: int):
    key = _emb_cache_key(corpus_path, model_name, max_len, expect_n)
    e, i, m, fx = _emb_cache_paths(key)
    if not (os.path.exists(e) and os.path.exists(i) and os.path.exists(m)):
        return None
    try:
        meta = json.load(open(m, "r", encoding="utf-8"))
        if meta.get("model_name") != model_name: return None
        ids = json.load(open(i, "r", encoding="utf-8"))
        embs = np.load(e, mmap_mode=("r" if USE_MEMMAP_ON_LOAD else None))
        if embs.shape[0] != len(ids): return None
        print(f"[CACHE] E5 embeddings restored from cache: {e} (shape={embs.shape}, memmap={USE_MEMMAP_ON_LOAD})")
        return {"embeddings": embs, "ids": ids, "meta": meta, "faiss_path": fx, "key": key}
    except Exception as ex:
        print("[CACHE] Failed to load cache:", ex)
        return None

def save_embeddings_cache(corpus_path: str, model_name: str, max_len: int, ids: List[str], embs: np.ndarray):
    key = _emb_cache_key(corpus_path, model_name, max_len, len(ids))
    e, i, m, fx = _emb_cache_paths(key)
    arr = np.asarray(embs, dtype=np.float32, order="C")
    np.save(e, arr)
    json.dump(list(ids), open(i, "w", encoding="utf-8"), ensure_ascii=False)
    meta = {"model_name": model_name, "num_documents": len(ids), "dim": int(arr.shape[1]),
            "corpus_path": str(corpus_path), "max_len": int(512)}
    json.dump(meta, open(m, "w", encoding="utf-8"))
    print(f"[CACHE] Saved embeddings: {e}  (ids: {i}, meta: {m})")
    return {"key": key, "emb_path": e, "ids_path": i, "meta_path": m, "faiss_path": fx}

# ---------------------- FAISS helpers (CPU-only) ----------------------
try:
    import faiss
    FAISS=True
except Exception as e:
    print("[FAISS] unavailable:", e)
    FAISS=False

def build_index(xb: np.ndarray):
    xb = np.asarray(xb, dtype=np.float32, order="C")
    idx = faiss.IndexFlatIP(xb.shape[1])   # CPU index
    idx.add(xb)
    print("[FAISS] Using CPU IndexFlatIP")
    return idx

def save_faiss_cpu_index(index, path: str):
    try:
        faiss.write_index(index, path)  # already CPU
        print(f"[CACHE] Saved FAISS CPU index → {path}")
    except Exception as e:
        print("[CACHE] Could not save FAISS index:", e)

def load_faiss_cpu(path: str):
    try:
        idx = faiss.read_index(path)
        print("[FAISS] Loaded CPU index.")
        return idx
    except Exception as e:
        print("[CACHE] Failed to load FAISS index:", e)
        return None

# ---------------------- Build / Load E5 corpus embeddings & index ----------------------
e5 = E5Retriever(E5_MODEL_NAME)
doc_ids = list(corpus.keys())
doc_texts_map = corpus  # uid -> text

# Try cache
cache = load_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, expect_n=len(doc_ids))
if cache is not None:
    cached_ids = cache["ids"]
    if len(cached_ids) == len(doc_ids):
        doc_ids = cached_ids
        doc_texts = [doc_texts_map[i] for i in doc_ids]
        X = cache["embeddings"]
        fx_path = cache["faiss_path"]
        if FAISS and os.path.exists(fx_path):
            index = load_faiss_cpu(fx_path)
        elif FAISS:
            index = build_index(X)
            save_faiss_cpu_index(index, fx_path)
        else:
            index = None
    else:
        print("[CACHE] Cached ids count mismatch; recomputing embeddings.")
        cache = None

if cache is None:
    doc_ids = list(corpus.keys())
    doc_texts = [corpus[d] for d in doc_ids]
    print("[E5] Embedding corpus …")
    X = e5.embed(doc_texts, is_query=False, bs=128)
    print(f"[E5] Corpus embeddings: {X.shape}")
    saved_meta = save_embeddings_cache(CORPUS_JSONL, E5_MODEL_NAME, 512, doc_ids, X)
    if FAISS:
        index = build_index(X)
        save_faiss_cpu_index(index, saved_meta["faiss_path"])
    else:
        index = None

# Fallback vector-matmul for search if FAISS missing
if not FAISS and isinstance(X, np.memmap):
    Xdot = np.array(X).T
elif not FAISS:
    Xdot = X.T

# ---------------------- TF-IDF (word) build ----------------------
print("[TFIDF] Fitting word TF-IDF on corpus …")
tfidf_vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATS,
                            ngram_range=(TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX))
tfidf_mat = tfidf_vec.fit_transform(doc_texts).astype(np.float32)  # CSR float32
print(f"[TFIDF] Matrix shape={tfidf_mat.shape} nnz={tfidf_mat.nnz:,}")

# ---------------------- TF-IDF (char) build ----------------------
if ENABLE_CHAR_TFIDF:
    print("[TFIDF-CHAR] Fitting char TF-IDF on corpus …")
    tfidf_char_vec = TfidfVectorizer(
        analyzer=TFIDF_CHAR_ANALYZER,
        ngram_range=(TFIDF_CHAR_MIN, TFIDF_CHAR_MAX),
        max_features=TFIDF_CHAR_MAX_FEATS,
        dtype=np.float32,
    )
    tfidf_char_mat = tfidf_char_vec.fit_transform(doc_texts).tocsr().astype(np.float32, copy=False)
    print(f"[TFIDF-CHAR] Matrix shape={tfidf_char_mat.shape} nnz={tfidf_char_mat.nnz:,}")
else:
    tfidf_char_vec, tfidf_char_mat = None, None
    print("[TFIDF-CHAR] Disabled.")

# ---------------------- RRF utilities ----------------------
def rrf_fuse(e5: List[Tuple[int, float]],
             lex: List[Tuple[int, float]],
             k: int,
             lex_c: List[Tuple[int, float]] = None) -> List[int]:
    """
    Fuse up to three ranked lists (indices in doc_ids) using weighted RRF.
    Accepts: e5 (pairs), lex word (pairs), optional lex char (pairs).
    """
    # Prepare rank maps
    sources = []
    weights = []
    if e5:
        sources.append({gi: r for r, (gi, _) in enumerate(e5)})
        weights.append(WEIGHT_E5)
    if lex:
        sources.append({gi: r for r, (gi, _) in enumerate(lex)})
        weights.append(WEIGHT_TFIDF)
    if lex_c:
        sources.append({gi: r for r, (gi, _) in enumerate(lex_c)})
        weights.append(WEIGHT_TFIDF_CHAR)

    if not sources:
        return []

    w_sum = sum(weights) if sum(weights) > 0 else 1.0
    norm_w = [w / w_sum for w in weights]

    universe = set().union(*[set(s.keys()) for s in sources])
    fused = []
    BIG = 10**9
    for gi in universe:
        s = 0.0
        for m, w in zip(sources, norm_w):
            r = m.get(gi, BIG)
            if r < BIG:
                s += float(w) / (RRF_K + r)
        fused.append((gi, s))

    fused.sort(key=lambda x: x[1], reverse=True)
    return [gi for gi, _ in fused[:k]]

# ---------------------- Per-source top-k (with scores) ----------------------
@torch.no_grad()
def e5_topk_with_scores(query: str, k: int):
    qv = e5.embed([query], is_query=True, bs=1).astype(np.float32)
    if FAISS:
        D, I = index.search(qv, min(k, len(doc_ids)))
        pairs = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i >= 0]
        return pairs  # already sorted by score desc
    else:
        sims = (qv @ Xdot)[0]
        ords = np.argsort(sims)[::-1][:k]
        return [(int(i), float(sims[i])) for i in ords]

def tfidf_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    qv = tfidf_vec.transform([query]).astype(np.float32)   # 1 x F (CSR float32)
    prod = (tfidf_mat @ qv.T)                              # N x 1 sparse
    scores = (prod.toarray().ravel()
              if hasattr(prod, "toarray") else np.asarray(prod).ravel())
    scores = scores.astype(np.float32, copy=False)
    take = min(k, scores.shape[0])
    if take <= 0:
        return []
    part = np.argpartition(scores, -take)[-take:]
    ords = part[np.argsort(scores[part])[::-1]]
    return [(int(i), float(scores[i])) for i in ords]

def tfidf_char_topk_with_scores(query: str, k: int) -> List[Tuple[int, float]]:
    """Sparse top-k on char TF-IDF (no dense materialization)."""
    if not ENABLE_CHAR_TFIDF or tfidf_char_vec is None or tfidf_char_mat is None or k <= 0:
        return []
    v = tfidf_char_vec.transform([query])      # 1 x F CSR
    s = (tfidf_char_mat @ v.T).tocoo()         # N x 1 sparse
    if s.nnz == 0:
        return []
    take = min(k, s.nnz)
    part = np.argpartition(s.data, -take)[-take:]
    ords = part[np.argsort(s.data[part])[::-1]]
    return [(int(s.row[i]), float(s.data[i])) for i in ords]

def fused_topk_ids(query: str, k_final: int) -> List[str]:
    pool_k = max(int(k_final * RRF_POOL_MULT), RRF_POOL_MIN)
    e5_list        = e5_topk_with_scores(query, pool_k)
    tfidf_list     = tfidf_topk_with_scores(query, pool_k)
    tfidf_char_list= tfidf_char_topk_with_scores(query, pool_k)
    fused_idx  = rrf_fuse(e5_list, tfidf_list, k_final, lex_c=tfidf_char_list)
    return [doc_ids[i] for i in fused_idx]

# ---------------------- Build top-K (fused) groups for every query ----------------------
# Each group: {'query', 'query_uuid', 'case_name', 'pids', 'texts', 'labels'}
groups = []
retrieved_relevant = 0
total_relevant = 0
queries_with_relevant = 0
for row in train_rows:
    pids   = fused_topk_ids(row["query"], K_E5)
    texts  = [doc_texts_map[pid] for pid in pids]
    labels = [int(row["gt"].get(pid, 0)) for pid in pids]  # unlabeled -> 0

    groups.append({
        "query": row["query"],
        "query_uuid": row.get("query_uuid"),
        "case_name": row.get("case_name"),
        "pids": pids,
        "texts": texts,
        "labels": labels
    })

    rel_total = sum(1 for v in row["gt"].values() if v > 0)
    if rel_total:
        queries_with_relevant += 1
        total_relevant += rel_total
        retrieved_relevant += sum(1 for pid in pids if row["gt"].get(pid, 0) > 0)

print(f"[GROUPS] built fused (E5 ⊕ TF-IDF[word]{' ⊕ TF-IDF[char]' if ENABLE_CHAR_TFIDF else ''}) top-{K_E5} for all queries.")
if total_relevant:
    recall_at_k = retrieved_relevant / total_relevant
    print(f"[METRICS] Recall@{K_E5} = {recall_at_k:.4f} ({retrieved_relevant}/{total_relevant}) "
          f"over {queries_with_relevant} queries with relevance.")
else:
    print(f"[METRICS] Recall@{K_E5} undefined (no relevant labels).")

# ---------------------- Free heavy objects to reduce RAM/VRAM ----------------------
try: del e5
except Exception: pass
try: del index
except Exception: pass
for _name in ("X", "Xdot", "doc_texts", "doc_texts_map", "tfidf_mat", "tfidf_vec",
              "tfidf_char_mat", "tfidf_char_vec"):
    if _name in globals(): globals()[_name] = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[CLEANUP] Freed E5/FAISS & TF-IDF; cleared CUDA cache.")

# ---------------------- Split by query + optional leakage filter ----------------------
if SPLIT_VAL:
    idxs = list(range(len(groups)))
    random.shuffle(idxs)
    cut = int(len(groups)*(1.0-VAL_SIZE))
    groups_train = [groups[i] for i in idxs[:cut]]
    groups_val   = [groups[i] for i in idxs[cut:]]
    print(f"[SPLIT] train groups={len(groups_train)}  val groups={len(groups_val)}  (VAL_SIZE={VAL_SIZE})")

    if FILTER_VAL_LEAKAGE:
        val_doc_set = set()
        for g in groups_val:
            val_doc_set.update(g["pids"])

        def filter_train_docs(train_groups, banned):
            kept, drop_items, drop_groups = [], 0, 0
            for g in train_groups:
                mask = [pid not in banned for pid in g["pids"]]
                if not any(mask):
                    drop_groups += 1
                    continue
                pids  = [p for p,m in zip(g["pids"], mask) if m]
                texts = [t for t,m in zip(g["texts"], mask) if m]
                labs  = [l for l,m in zip(g["labels"],mask) if m]
                drop_items += (len(g["pids"]) - len(pids))
                kept.append({
                    "query": g["query"],
                    "query_uuid": g.get("query_uuid"),
                    "case_name": g.get("case_name"),
                    "pids": pids, "texts": texts, "labels": labs
                })
            return kept, drop_items, drop_groups

        groups_train, n_drop_items, n_drop_groups = filter_train_docs(groups_train, val_doc_set)
        print(f"[LEAKAGE] removed {n_drop_items} train items; dropped {n_drop_groups} fully-overlapping train groups")
    else:
        print("[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).")
else:
    groups_train = groups
    groups_val   = []
    print(f"[SPLIT] no validation split (SPLIT_VAL=0): using ALL {len(groups_train)} groups for training.")

# ---------------------- Save stage-1 outputs ----------------------
def _write_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

_write_jsonl(GROUPS_TRAIN_JSONL, groups_train)
_write_jsonl(GROUPS_VAL_JSONL, groups_val)

meta = {
    # include both keys so downstream code can read either
    "K_FINAL": K_E5,
    "K_E5": K_E5,
    "VAL_SIZE": VAL_SIZE, "SEED": SEED,
    "split_val": int(SPLIT_VAL), "filter_val_leakage": int(FILTER_VAL_LEAKAGE),
    "corpus_path": CORPUS_JSONL, "train_path": TRAIN_JSONL,
    "e5_model": E5_MODEL_NAME, "groups_train": len(groups_train), "groups_val": len(groups_val),
    "fusion": {
        "type": "RRF",
        "RRF_K": RRF_K,
        "RRF_POOL_MULT": RRF_POOL_MULT,
        "RRF_POOL_MIN": RRF_POOL_MIN,
        "weights": {
            "e5": WEIGHT_E5,
            "tfidf_word": WEIGHT_TFIDF,
            "tfidf_char": WEIGHT_TFIDF_CHAR
        }
    },
    "tfidf": {
        "word": {"max_features": TFIDF_MAX_FEATS, "ngram": [TFIDF_NGRAM_MIN, TFIDF_NGRAM_MAX]},
        "char": {
            "enabled": int(ENABLE_CHAR_TFIDF),
            "analyzer": TFIDF_CHAR_ANALYZER,
            "ngram": [TFIDF_CHAR_MIN, TFIDF_CHAR_MAX],
            "max_features": TFIDF_CHAR_MAX_FEATS
        }
    }
}
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"[SAVE] groups_train → {GROUPS_TRAIN_JSONL}")
print(f"[SAVE] groups_val   → {GROUPS_VAL_JSONL}")
print(f"[SAVE] meta         → {META_JSON}")


[CONFIG] device=cuda  K_FINAL=70  VAL_SIZE=0.15  SEED=42  SPLIT_VAL=1  FILTER_VAL_LEAKAGE=0
[PATHS] CORPUS=/content/hsrc_corpus.jsonl  TRAIN=/content/hsrc_train_augmented.jsonl  OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16  CACHE=/content/e5_cache  STAGE1=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1
[FUSION] RRF_POOL_MULT=3.0  RRF_POOL_MIN=150  RRF_K=60  WEIGHTS: E5=0.6 TFIDF(word)=0.4 TFIDF(char)=0.15
[TFIDF(word)] max_features=300000 ngram=(1,2)
[TFIDF(char)] enabled=True analyzer=char_wb ngram=(3,4) max_features=200000
[DATA] corpus docs=127,731
[DATA] queries total=2,034


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

[E5] Embedding corpus …
[E5] Corpus embeddings: (127731, 1024)
[CACHE] Saved embeddings: /content/e5_cache/e5_1f10598cb43dba59_embeddings.npy  (ids: /content/e5_cache/e5_1f10598cb43dba59_ids.json, meta: /content/e5_cache/e5_1f10598cb43dba59_meta.json)
[FAISS] Using CPU IndexFlatIP
[CACHE] Saved FAISS CPU index → /content/e5_cache/e5_1f10598cb43dba59_index.faiss
[TFIDF] Fitting word TF-IDF on corpus …
[TFIDF] Matrix shape=(127731, 300000) nnz=21,476,500
[TFIDF-CHAR] Fitting char TF-IDF on corpus …
[TFIDF-CHAR] Matrix shape=(127731, 200000) nnz=106,512,380
[GROUPS] built fused (E5 ⊕ TF-IDF[word] ⊕ TF-IDF[char]) top-70 for all queries.
[METRICS] Recall@70 = 0.8328 (12732/15289) over 2021 queries with relevance.
[CLEANUP] Freed E5/FAISS & TF-IDF; cleared CUDA cache.
[SPLIT] train groups=1728  val groups=306  (VAL_SIZE=0.15)
[LEAKAGE] Skipping leakage removal — train may include docs also in val (submission-style).
[SAVE] groups_train → /content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1/grou

## Step 3: Train Per-Corpus Cross-Encoders

Fine-tunes **BGE-reranker-v2-m3** separately for each corpus (wiki, kz, knesset).

**Loss**: ListNet (listwise ranking loss aligned with NDCG optimization)

**Per-corpus hyperparameters**:
| Corpus | Epochs | LR | Max Length | τ |
|--------|--------|------|------------|-----|
| Wiki | 2 | 8e-6 | 320 | 0.9 |
| KZ | 3 | 8e-6 | 448 | 1.1 |
| Knesset | 4 | 8e-6 | 496 | 0.9 |

Saves best checkpoint (by validation NDCG@20) in FP16 format.

In [7]:
# === Cell 2: Stage 2 — fine-tune corpus-specific cross-encoders with per-corpus params ===
import os, json, math, time, random, gc
from pathlib import Path
from typing import List, Dict, Tuple, Union
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup

# ---------------------- Base Config (defaults, can be overridden per corpus) ----------------------
OUTPUT_DIR      = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")
CE_MODEL_NAME   = os.getenv("CE_MODEL_NAME","BAAI/bge-reranker-v2-m3")   # default CE for all

# Training/eval defaults (per-corpus overrides below)
EPOCHS          = int(os.getenv("EPOCHS", "2"))
LR              = float(os.getenv("LR", "1.5e-5"))
BATCH_GROUPS    = int(os.getenv("BATCH_GROUPS", "2"))
GRAD_ACCUM      = int(os.getenv("GRAD_ACCUM", "1"))
MAX_LEN         = int(os.getenv("MAX_LEN", "384"))
WEIGHT_DECAY    = float(os.getenv("WEIGHT_DECAY", "0.02"))
CLIP_NORM       = float(os.getenv("CLIP_NORM", "1.0"))
TAU             = float(os.getenv("TAU", "1.1"))
VAL_EVAL_K      = int(os.getenv("VAL_EVAL_K", "20"))

# Eval/pretokenization
EVAL_BATCH_PAIRS   = int(os.getenv("EVAL_BATCH_PAIRS", "128"))
PAD_TO_MULTIPLE_OF = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))

SEED = int(os.getenv("SEED", "42"))

# Stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
# These may be updated automatically from stage1 meta below
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k70.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k70.jsonl"))

# ---------------------- Which corpora to run ----------------------
# "all" or any subset list, e.g., ["kz"], ["kz","knesset"], ["wiki"]
RUN_CORPORA = "all"   # <-- change here in notebook if you want a subset

# ---------------------- Per-corpus overrides ----------------------
# Put any field here to override defaults for that corpus.
# Supported keys include: CE_MODEL_NAME, EPOCHS, LR, BATCH_GROUPS, GRAD_ACCUM, MAX_LEN,
# WEIGHT_DECAY, CLIP_NORM, TAU, VAL_EVAL_K, EVAL_BATCH_PAIRS, PAD_TO_MULTIPLE_OF
PER_CORPUS_OVERRIDES = {
  # Wikipedia: short passages, many label-4s, already strong baseline → keep it efficient & stable
  "wiki": {
    "EPOCHS": 2,
    "LR": 8e-6,
    "MAX_LEN": 320,          # passages: mean≈134, p95≈238 → 320 comfortably covers
    "BATCH_GROUPS": 2,
    "GRAD_ACCUM": 1,
    "TAU": 0.9,
    "EVAL_BATCH_PAIRS": 192, # faster eval on the A10
  },

  # KZ (Kol-Zchut): longest passages + heavy positives tail → give more context & a touch more temp
  "kz":  {
    "EPOCHS": 3,
    "LR": 8e-6,
    "MAX_LEN": 448,
    "BATCH_GROUPS": 2,
    "GRAD_ACCUM": 1,
    "TAU": 1.1,
    "EVAL_BATCH_PAIRS": 192,
  },


  # Knesset: hardest domain, medium length, fewer label-4s → steadier LR, one extra epoch, slightly higher τ
  "knesset":{
    "EPOCHS": 4,
    "LR": 8e-6,
    "MAX_LEN": 496,
    "BATCH_GROUPS": 2,
    "GRAD_ACCUM": 1,
    "TAU": 0.9,
    "EVAL_BATCH_PAIRS": 128,
  },
}

# ---------------------- Corpus keys & aliases ----------------------
CORPUS_KEYS = {
    "mafat_retrieval_wikipedia_corpus": "wiki",
    "mafat_retrieval_kz_corpus":        "kz",
    "mafat_retrieval_knesset_corpus":   "knesset",
}
SLUG_TO_CORPUSKEY = {v: k for k, v in CORPUS_KEYS.items()}
# Accept a common misspelling
ALIASES = {"kenesset": "knesset"}

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Allow TF32 for speed on Ampere
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"[CONFIG] device={device}")
print(f"[PATHS] OUT={OUTPUT_DIR}  STAGE1={STAGE1_DIR}")

# ---------------------- IO ----------------------
def _read_jsonl(path):
    items=[]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

# Auto-match Stage-1 K for group paths if meta exists (support both K_E5 and K_FINAL)
_meta_path = os.path.join(STAGE1_DIR, "meta.json")
_stage1_meta = None
if os.path.exists(_meta_path):
    try:
        _stage1_meta = json.load(open(_meta_path, "r", encoding="utf-8"))
        _k_meta = _stage1_meta.get("K_E5") or _stage1_meta.get("K_FINAL")
        if _k_meta:
            if "GROUPS_TRAIN_JSONL" not in os.environ:
                GROUPS_TRAIN_JSONL = os.path.join(STAGE1_DIR, f"groups_train_k{_k_meta}.jsonl")
            if "GROUPS_VAL_JSONL" not in os.environ:
                GROUPS_VAL_JSONL   = os.path.join(STAGE1_DIR, f"groups_val_k{_k_meta}.jsonl")
    except Exception:
        pass

groups_train = _read_jsonl(GROUPS_TRAIN_JSONL)
groups_val   = _read_jsonl(GROUPS_VAL_JSONL)
print(f"[DATA] train groups={len(groups_train)}  val groups={len(groups_val)}")

# Ensure groups have case_name; Stage-1 updated to include it, but fall back if needed
def _q2case_from_train_json(stage1_meta):
    mp = {}
    tp = stage1_meta.get("train_path") if stage1_meta else None
    if tp and os.path.exists(tp):
        with open(tp, "r", encoding="utf-8") as f:
            for ln in f:
                if not ln.strip(): continue
                o = json.loads(ln)
                q = o.get("query",""); cn = o.get("case_name") or "unknown"
                if q and q not in mp: mp[q] = cn
    return mp

_q2case = _q2case_from_train_json(_stage1_meta)

def _attach_case(groups):
    for g in groups:
        if "case_name" not in g or not g["case_name"]:
            g["case_name"] = _q2case.get(g.get("query",""), "unknown")
    return groups

groups_train = _attach_case(groups_train)
groups_val   = _attach_case(groups_val)

# Skip groups with zero positives (speeds training)
_before = len(groups_train)
groups_train = [g for g in groups_train if any(l > 0 for l in g["labels"])]
print(f"[DATA] filtered train groups with no positives: {_before - len(groups_train)} dropped; {len(groups_train)} remain")

# ---------------------- Dataset / Collate ----------------------
class ListwiseDataset(torch.utils.data.Dataset):
    def __init__(self, groups): self.groups = groups
    def __len__(self): return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return g["query"], g["texts"], g["labels"], g["pids"]

def collate_listwise(batch, tokenizer, max_len=512):
    Q, P, gains, spans, PIDS = [], [], [], [], []
    cur = 0
    for q, texts, labs, pids in batch:
        Q += [q]*len(texts)
        P += texts
        gains += [float((2**int(l))-1) for l in labs]
        PIDS += pids
        spans.append((cur, cur+len(texts)))
        cur += len(texts)
    enc = tokenizer(Q, P, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return enc, torch.tensor(gains, dtype=torch.float32), spans, PIDS

# ---------------------- Loss ----------------------
def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0:
            continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ---------------------- Eval helpers ----------------------
def _dcg_at_k(rels, k=20):
    return sum(((2**r-1)/math.log2(i+2) for i,r in enumerate(rels[:k])), 0.0)

def ndcg_at_k(ids_sorted: List[str], gt_map: Dict[str,int], k=20):
    rels = [gt_map.get(pid, 0) for pid in ids_sorted[:k]]
    dcg  = _dcg_at_k(rels, k)
    idcg = _dcg_at_k(sorted(gt_map.values(), reverse=True), k)
    return 0.0 if idcg==0.0 else float(dcg/idcg)

@torch.no_grad()
def evaluate_ndcg_groups(model, loader_or_pretok: Union[dict, torch.utils.data.DataLoader], k=20) -> List[float]:
    model.eval()
    # Fast pre-tokenized path
    if isinstance(loader_or_pretok, dict) and "enc" in loader_or_pretok:
        enc_full = loader_or_pretok["enc"]
        spans    = loader_or_pretok["spans"]
        rels_all = loader_or_pretok["rels"]

        if not spans or rels_all.numel() == 0:
            return []

        N = enc_full["input_ids"].size(0)
        scores = torch.empty(N, dtype=torch.float32)

        start = 0
        while start < N:
            end = min(start + params_runtime["EVAL_BATCH_PAIRS"], N)  # uses runtime params (set below)
            sl = slice(start, end)
            inputs = {k: v[sl].to(device, non_blocking=True) for k, v in enc_full.items()}
            ctx = (torch.autocast(device_type="cuda", dtype=torch.float16)
                   if device=="cuda" else torch.cpu.amp.autocast(enabled=False))
            with torch.inference_mode(), ctx:
                logits = model(**inputs).logits
                if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
                elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
                else: s = logits.view(-1).float()
            scores[sl] = s.detach().cpu()
            start = end

        denom = 1.0 / np.log2(np.arange(2, params_runtime["VAL_EVAL_K"] + 2))
        s_np = scores.numpy(); r_np = rels_all.numpy()
        ndcgs = []
        for (st, ed) in spans:
            group_scores = s_np[st:ed]
            group_rels   = r_np[st:ed]
            if group_rels.size == 0:
                ndcgs.append(0.0); continue
            order = np.argsort(-group_scores)
            rel_sorted = group_rels[order][:params_runtime["VAL_EVAL_K"]]
            dcg = ((np.power(2.0, rel_sorted, dtype=np.float64) - 1.0) * denom[:len(rel_sorted)]).sum()
            ideal = np.sort(group_rels)[::-1][:params_runtime["VAL_EVAL_K"]]
            idcg = ((np.power(2.0, ideal, dtype=np.float64) - 1.0) * denom[:len(ideal)]).sum()
            ndcgs.append(0.0 if idcg <= 0.0 else float(dcg / idcg))
        return ndcgs

    # Slow path (kept for completeness)
    ndcgs=[]
    for enc_cpu, gains_cpu, spans, pids in loader_or_pretok:
        enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        logits = model(**enc).logits
        if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
        elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
        else: s = logits.view(-1).float()
        scores = s.detach().cpu().numpy().tolist()
        for (st,ed) in spans:
            group_pids   = pids[st:ed]
            group_scores = scores[st:ed]
            g_local      = gains_cpu[st:ed].tolist()
            rels_local = []
            for g in g_local:
                if g <= 0: rels_local.append(0)
                elif math.isclose(g,1.0):  rels_local.append(1)
                elif math.isclose(g,3.0):  rels_local.append(2)
                elif math.isclose(g,7.0):  rels_local.append(3)
                elif math.isclose(g,15.0): rels_local.append(4)
                else: rels_local.append(int(round(math.log2(g+1))))
            gt = {pid: rel for pid, rel in zip(group_pids, rels_local)}
            ranked = [pid for pid,_ in sorted(zip(group_pids, group_scores), key=lambda x:x[1], reverse=True)]
            ndcgs.append(ndcg_at_k(ranked, gt, k=params_runtime["VAL_EVAL_K"]))
    return ndcgs

def _mean_or_zero(vals): return float(np.mean(vals)) if len(vals) > 0 else 0.0

def build_val_pretok(groups, tokenizer, max_len=512, pad_multi=8):
    all_input_ids, all_attn, all_ttids = [], [], []
    all_rels, all_pids, spans = [], [], []
    cur = 0
    for g in groups:
        q, texts, labels, pids = g["query"], g["texts"], g["labels"], g["pids"]
        enc = tokenizer(
            [q] * len(texts), texts,
            padding="max_length", truncation=True, max_length=max_len,
            pad_to_multiple_of=pad_multi, return_tensors="pt",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
        if "token_type_ids" in enc: all_ttids.append(enc["token_type_ids"])
        all_rels.extend([int(l) for l in labels]); all_pids.extend(pids)
        spans.append((cur, cur + len(texts))); cur += len(texts)

    if all_input_ids:
        input_ids = torch.cat(all_input_ids, dim=0)
        attention_mask = torch.cat(all_attn, dim=0)
        enc_full = {"input_ids": input_ids, "attention_mask": attention_mask}
        if len(all_ttids) > 0: enc_full["token_type_ids"] = torch.cat(all_ttids, dim=0)
        for k in list(enc_full.keys()):
            enc_full[k] = enc_full[k].pin_memory()
        rels = torch.tensor(all_rels, dtype=torch.int16)
    else:
        L = max_len
        enc_full = {"input_ids": torch.empty(0, L, dtype=torch.long),
                    "attention_mask": torch.empty(0, L, dtype=torch.long)}
        rels = torch.empty(0, dtype=torch.int16)
    return {"enc": enc_full, "spans": spans, "pids": all_pids, "rels": rels}

# ---------------------- Helpers: save size ----------------------
def _dir_size_bytes(path: str) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try: total += os.path.getsize(fp)
            except OSError: pass
    return total

def _human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    i = 0; v = float(n)
    while v >= 1024.0 and i < len(units)-1: v /= 1024.0; i += 1
    return f"{v:.2f} {units[i]}"

def _save_fp16_and_report(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    orig_dtype = next(model.parameters()).dtype
    try:
        model.to(dtype=torch.float16)
        model.save_pretrained(out_dir, safe_serialization=True)  # safetensors
        tokenizer.save_pretrained(out_dir)
    finally:
        model.to(dtype=orig_dtype)
    sz = _dir_size_bytes(out_dir)
    print(f"[SAVE] Checkpoint saved (fp16) → {out_dir}  |  folder size = {_human_bytes(sz)}")

# ---------------------- Filters & param plumbing ----------------------
def _filter_by_corpus(groups, corpus_key: str):
    return [g for g in groups if (g.get("case_name") == corpus_key)]

def _resolve_run_corpora(run_spec):
    if isinstance(run_spec, str):
        run_spec = run_spec.strip().lower()
        if run_spec == "all":
            return list(SLUG_TO_CORPUSKEY.keys())
        run_spec = ALIASES.get(run_spec, run_spec)
        return [run_spec] if run_spec in SLUG_TO_CORPUSKEY else []
    elif isinstance(run_spec, (list, tuple, set)):
        slugs = []
        for x in run_spec:
            y = ALIASES.get(str(x).lower(), str(x).lower())
            if y in SLUG_TO_CORPUSKEY: slugs.append(y)
        return list(dict.fromkeys(slugs))  # de-dup, preserve order
    else:
        return []

BASE_PARAMS = {
    "CE_MODEL_NAME": CE_MODEL_NAME,
    "EPOCHS": EPOCHS,
    "LR": LR,
    "BATCH_GROUPS": BATCH_GROUPS,
    "GRAD_ACCUM": GRAD_ACCUM,
    "MAX_LEN": MAX_LEN,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "CLIP_NORM": CLIP_NORM,
    "TAU": TAU,
    "VAL_EVAL_K": VAL_EVAL_K,
    "EVAL_BATCH_PAIRS": EVAL_BATCH_PAIRS,
    "PAD_TO_MULTIPLE_OF": PAD_TO_MULTIPLE_OF,
}

def _params_for_slug(slug: str) -> Dict[str, Union[int, float, str]]:
    p = BASE_PARAMS.copy()
    if slug in PER_CORPUS_OVERRIDES:
        for k, v in PER_CORPUS_OVERRIDES[slug].items():
            p[k] = v
    return p

# ---------------------- Train ONE corpus-specific CE ----------------------
def train_one_corpus(slug: str) -> Dict[str, float]:
    corpus_key = SLUG_TO_CORPUSKEY[slug]
    out_dir = os.path.join(OUTPUT_DIR, f"ce_{slug}")
    os.makedirs(out_dir, exist_ok=True)

    # Hydrate per-corpus params
    global params_runtime
    params_runtime = _params_for_slug(slug)  # used inside eval fn as well
    print(f"\n[SETUP:{slug}] params={params_runtime}")

    # Split groups by corpus
    tr = _filter_by_corpus(groups_train, corpus_key)
    va = _filter_by_corpus(groups_val,   corpus_key)
    print(f"[DATA:{slug}] train={len(tr)}  val={len(va)}  out={out_dir}")
    if len(tr) == 0 or len(va) == 0:
        print(f"[WARN:{slug}] Skipping (no data).")
        return {"baseline": 0.0, "best": 0.0}

    tok = AutoTokenizer.from_pretrained(params_runtime["CE_MODEL_NAME"])
    # Fresh model per corpus
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            params_runtime["CE_MODEL_NAME"], trust_remote_code=True, attn_implementation="sdpa"
        ).to(device)
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(
            params_runtime["CE_MODEL_NAME"], trust_remote_code=True
        ).to(device)
    model = model.float()

    # Enable gradient checkpointing
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    if hasattr(model, "config") and hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    # Loaders
    train_ds = ListwiseDataset(tr)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=params_runtime["BATCH_GROUPS"], shuffle=True,
        collate_fn=lambda b: collate_listwise(b, tok, params_runtime["MAX_LEN"]),
        pin_memory=torch.cuda.is_available(), num_workers=2 if torch.cuda.is_available() else 0,
        persistent_workers=torch.cuda.is_available()
    )

    pretok_val = build_val_pretok(va, tok, max_len=params_runtime["MAX_LEN"],
                                  pad_multi=params_runtime["PAD_TO_MULTIPLE_OF"])

    # Optim / sched
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params_runtime["LR"],
        weight_decay=params_runtime["WEIGHT_DECAY"], eps=1e-8, betas=(0.9, 0.999)
    )
    updates_per_epoch = max(1, math.ceil(len(train_loader) / max(1, params_runtime["GRAD_ACCUM"])))
    num_update_steps  = max(1, updates_per_epoch * params_runtime["EPOCHS"])
    warmup_ratio = 0.10
    warmup = max(1, int(warmup_ratio * num_update_steps))
    lr_end_frac = 0.10
    lr_end = max(params_runtime["LR"] * lr_end_frac, 1e-8)
    scheduler = get_polynomial_decay_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup,
        num_training_steps=num_update_steps,
        lr_end=lr_end,
        power=1.0
    )

    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # Baseline
    print(f"\n[VAL:{slug}] evaluating pretrained CE on VAL …")
    baseline_list = evaluate_ndcg_groups(model, pretok_val, k=params_runtime["VAL_EVAL_K"])
    baseline = _mean_or_zero(baseline_list)
    print(f"[VAL:{slug}] Baseline NDCG@{params_runtime['VAL_EVAL_K']}: {baseline:.4f}")

    best = baseline
    compute_loss = listnet_loss

    for ep in range(1, params_runtime["EPOCHS"]+1):
        model.train()
        running, micro = 0.0, 0
        t0 = time.perf_counter()

        for step, (enc_cpu, gains_cpu, spans, _pids) in enumerate(train_loader, 1):
            enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
            gains = gains_cpu.to(device, non_blocking=True)

            if use_amp:
                ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
            else:
                class _NoOp:
                    def __enter__(self): pass
                    def __exit__(self, *args): return False
                ctx = _NoOp()

            with ctx:
                logits = model(**enc).logits
                if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
                elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
                else: s = logits.view(-1).float()
                loss = compute_loss(s, gains, spans, tau=params_runtime["TAU"])

            micro += 1
            if scaler is not None and use_amp:
                scaler.scale(loss).backward()
                if params_runtime["CLIP_NORM"] > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), params_runtime["CLIP_NORM"])
                if step % params_runtime["GRAD_ACCUM"] == 0:
                    scaler.step(optimizer); scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            else:
                loss.backward()
                if params_runtime["CLIP_NORM"] > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), params_runtime["CLIP_NORM"])
                if step % params_runtime["GRAD_ACCUM"] == 0:
                    optimizer.step(); scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            running += float(loss.item())
            if step % 50 == 0 or step == len(train_loader):
                elapsed = time.perf_counter() - t0
                print(f"[train:{slug} e{ep}/{params_runtime['EPOCHS']}] step {step}/{len(train_loader)} "
                      f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                      f"steps/s={step/max(1e-6,elapsed):.2f}")

        # Validate for this corpus
        val_list = evaluate_ndcg_groups(model, pretok_val, k=params_runtime["VAL_EVAL_K"])
        val_ndcg = _mean_or_zero(val_list)
        print(f"[VAL:{slug}] epoch {ep}  NDCG@{params_runtime['VAL_EVAL_K']}={val_ndcg:.4f}  (baseline {baseline:.4f})")

        if val_ndcg > best + 1e-4:
            best = val_ndcg
            _save_fp16_and_report(model, tok, out_dir)

    print(f"\n[RESULT:{slug}] Best VAL NDCG@{params_runtime['VAL_EVAL_K']}: {best:.4f} (baseline {baseline:.4f})")

    # Cleanup GPU RAM before next corpus
    del model, tok, optimizer, scheduler, scaler, train_ds, train_loader, pretok_val
    torch.cuda.empty_cache(); gc.collect()

    return {"baseline": float(baseline), "best": float(best)}

# ---------------------- Run training for selected corpora ----------------------
slugs_to_run = _resolve_run_corpora(RUN_CORPORA)
if not slugs_to_run:
    print("[WARN] Nothing to run. Set RUN_CORPORA = 'all' or a list like ['kz', 'knesset', 'wiki'].")
else:
    print(f"[RUN] corpora: {slugs_to_run}")

summary = {}
for slug in slugs_to_run:
    res = train_one_corpus(slug)
    summary[slug] = res

print("\n================ SUMMARY (per-corpus) ================")
for slug, v in summary.items():
    print(f"{slug:10s}  baseline={v['baseline']:.4f}  best={v['best']:.4f}")
print("======================================================")


[CONFIG] device=cuda
[PATHS] OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16  STAGE1=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1
[DATA] train groups=1728  val groups=306
[DATA] filtered train groups with no positives: 41 dropped; 1687 remain
[RUN] corpora: ['wiki', 'kz', 'knesset']

[SETUP:wiki] params={'CE_MODEL_NAME': 'BAAI/bge-reranker-v2-m3', 'EPOCHS': 2, 'LR': 8e-06, 'BATCH_GROUPS': 2, 'GRAD_ACCUM': 1, 'MAX_LEN': 320, 'WEIGHT_DECAY': 0.02, 'CLIP_NORM': 1.0, 'TAU': 0.9, 'VAL_EVAL_K': 20, 'EVAL_BATCH_PAIRS': 192, 'PAD_TO_MULTIPLE_OF': 8}
[DATA:wiki] train=626  val=116  out=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/ce_wiki

[VAL:wiki] evaluating pretrained CE on VAL …
[VAL:wiki] Baseline NDCG@20: 0.7442
[train:wiki e1/2] step 50/313 loss(avg)=3.0485  lr=6.45e-06 steps/s=1.08
[train:wiki e1/2] step 100/313 loss(avg)=2.8092  lr=7.51e-06 steps/s=1.08
[train:wiki e1/2] step 150/313 loss(avg)=2.7669  lr=6.88e-06 steps/s=1.08
[train:wiki e1/2] step 200/313 loss(avg)=2.7176  lr=6.24e-

| Corpus | Baseline | Best |
|--------|----------|------|
| Wiki | 0.7442 | 0.7577 |
| KZ | 0.5157 | 0.6027 |
| Knesset | 0.4751 | 0.5379 |